<div dir="rtl">

# جلسه ۲۵: Pandas ۳
## Pandas III

**بخش ۳: محاسبات علمی و کار با داده** — دوره‌ی پایتون برای علوم داده

مدرس: دکتر حسین حق‌بین

</div>

<div dir="rtl">

## 🎯 اهداف جلسه

در پایان این جلسه باید بتوانید:

- چند جدول را با <code dir="ltr">concat</code> **روی هم** یا **کنار هم** بچینید و پیش‌بینی کنید کجا <code dir="ltr">NaN</code> ظاهر می‌شود،
- دو جدول را با <code dir="ltr">merge</code> و <code dir="ltr">join</code> بر اساس یک **کلید** به هم وصل کنید و تعداد سطرهای نتیجه را برای <code dir="ltr">inner</code>، <code dir="ltr">left</code>، <code dir="ltr">right</code> و <code dir="ltr">outer</code> پیش‌بینی کنید،
- خطر **کلید تکراری** (انفجار تعداد سطرها) را بشناسید و با <code dir="ltr">validate</code> و <code dir="ltr">indicator</code> جلوی خطاهای بی‌صدا را بگیرید،
- تاریخ‌ها را با <code dir="ltr">pd.to&#95;datetime</code> تبدیل کنید، با <code dir="ltr">.dt</code> اجزای آن را بیرون بکشید و یک **سری زمانی** را برش بزنید،
- با <code dir="ltr">resample</code>، <code dir="ltr">rolling</code>، <code dir="ltr">shift</code>، <code dir="ltr">diff</code> و <code dir="ltr">pct&#95;change</code> ویژگی‌های زمانی بسازید، بدون اینکه اطلاعات **آینده** به مدل نشت کند.

> 📖 روش خواندن این جزوه و معنی علامت‌ها (💡 🔍 ❓ ⚠️ 📌 📝) در ابتدای جلسه‌ی ۱ توضیح داده شده است.

</div>

<div dir="rtl">

## فهرست مطالب

1. concat
2. merge و join
3. داده‌های زمانی

</div>

In [ ]:
import numpy as np
import pandas as pd

<div dir="rtl">

## ۱. concat

### 💡 مفهوم

در عمل داده‌ها تقریباً هیچ‌وقت در یک جدول آماده نیستند: فروش هر ماه در یک فایل جداست، ویژگی‌ها و برچسب‌ها جدا نگه‌داری می‌شوند، یا اطلاعات مشتری و سفارش در دو جدول مختلف‌اند. این جلسه درباره‌ی **ترکیب جدول‌ها**ست.

<code dir="ltr">pd.concat</code> ساده‌ترین ابزار است: چند <code dir="ltr">DataFrame</code> (یا <code dir="ltr">Series</code>) را در یک لیست می‌گیرد و آن‌ها را

- با <code dir="ltr">axis=0</code> (پیش‌فرض) **روی هم** می‌چیند (سطرها اضافه می‌شوند)،
- با <code dir="ltr">axis=1</code> **کنار هم** می‌چیند (ستون‌ها اضافه می‌شوند).

<code dir="ltr">concat</code> هیچ کلیدی را مقایسه نمی‌کند؛ فقط بر اساس **نام ستون‌ها** (در <code dir="ltr">axis=0</code>) یا **اندیس سطرها** (در <code dir="ltr">axis=1</code>) چیدمان را انجام می‌دهد.

</div>

In [ ]:
jan = pd.DataFrame({"city": ["Bushehr", "Shiraz"], "sales": [120, 95]})
feb = pd.DataFrame({"city": ["Bushehr", "Tehran"], "sales": [130, 210]})

print(pd.concat([jan, feb]))
print()
print(pd.concat([jan, feb], ignore_index=True))
print()
print(pd.concat([jan, feb], keys=["jan", "feb"]))

<div dir="rtl">

### 🔍 کد را بخوان

- در خروجی اول، اندیس‌ها <code dir="ltr">0, 1, 0, 1</code> هستند: <code dir="ltr">concat</code> اندیس هر جدول را **همان‌طور که هست** نگه می‌دارد. اندیس تکراری بعداً در <code dir="ltr">loc</code> (جلسه‌ی ۲۳) دردسر درست می‌کند: <code dir="ltr">result.loc[0]</code> دو سطر برمی‌گرداند.
- <code dir="ltr">ignore&#95;index=True</code> اندیس‌های قدیمی را دور می‌ریزد و یک اندیس تازه‌ی <code dir="ltr">0</code> تا <code dir="ltr">n - 1</code> می‌سازد. وقتی اندیس معنای خاصی ندارد، معمولاً همین را می‌خواهید.
- <code dir="ltr">keys</code> یک سطح اضافه به اندیس می‌دهد (یک **MultiIndex**) تا معلوم باشد هر سطر از کدام جدول آمده است؛ مثلاً <code dir="ltr">result.loc["feb"]</code> فقط سطرهای فوریه را برمی‌گرداند.

### 💡 ستون‌های ناهمسان

اگر جدول‌ها ستون‌های یکسانی نداشته باشند، <code dir="ltr">concat</code> به‌طور پیش‌فرض **اجتماع** ستون‌ها را نگه می‌دارد (<code dir="ltr">join="outer"</code>) و جاهای خالی را <code dir="ltr">NaN</code> می‌گذارد. با <code dir="ltr">join="inner"</code> فقط ستون‌های **مشترک** می‌مانند؛ همان اجتماع و اشتراک مجموعه‌ها در جلسه‌ی ۹.

</div>

In [ ]:
mar = pd.DataFrame({"city": ["Shiraz"], "sales": [101], "returns": [3]})

print(pd.concat([jan, mar], ignore_index=True))
print()
print(pd.concat([jan, mar], ignore_index=True, join="inner"))

<div dir="rtl">

🔍 جدول <code dir="ltr">jan</code> ستون <code dir="ltr">returns</code> ندارد؛ پس در نتیجه‌ی اول، دو سطر اول در این ستون <code dir="ltr">NaN</code> گرفته‌اند. دقت کنید که ستون <code dir="ltr">returns</code> به <code dir="ltr">float</code> تبدیل شده (<code dir="ltr">3.0</code>): <code dir="ltr">NaN</code> خودش یک عدد اعشاری است، پس ستونی که <code dir="ltr">NaN</code> دارد نمی‌تواند <code dir="ltr">int</code> بماند. (درباره‌ی رفتار با مقدارهای گمشده در جلسه‌ی ۲۶ مفصل صحبت می‌کنیم.)

### 💡 کنار هم چیدن: <code dir="ltr">axis=1</code>

با <code dir="ltr">axis=1</code>، سطرها بر اساس **اندیس** با هم جفت می‌شوند. کاربرد رایج: چسباندن ستون برچسب به جدول ویژگی‌ها.

</div>

In [ ]:
X = pd.DataFrame({"age": [25, 32, 47]})
y = pd.Series([0, 1, 1], name="label")
print(pd.concat([X, y], axis=1))

<div dir="rtl">

### ⚠️ اشتباه رایج: اندیس‌های ناهماهنگ در <code dir="ltr">axis=1</code>

<code dir="ltr">concat</code> با <code dir="ltr">axis=1</code> به **اندیس** نگاه می‌کند، نه به ترتیب سطرها. اگر یکی از دو جدول قبلاً فیلتر یا مرتب شده و اندیسش با دیگری فرق دارد، سطرها **اشتباه** جفت می‌شوند یا <code dir="ltr">NaN</code> ظاهر می‌شود؛ بدون هیچ پیام خطایی.

### ❓ پیش‌بینی کن

قبل از اجرا: نتیجه چند سطر دارد و کجا <code dir="ltr">NaN</code> دارد؟

</div>

In [ ]:
features = pd.DataFrame({"x": [1.5, 2.0, 3.5]}, index=[10, 11, 12])
labels = pd.Series([1, 0, 1], index=[11, 12, 13], name="y")
print(pd.concat([features, labels], axis=1))

<div dir="rtl">

🔍 اندیس‌ها <code dir="ltr">{10, 11, 12}</code> و <code dir="ltr">{11, 12, 13}</code> هستند و نتیجه **اجتماع** آن‌هاست: چهار سطر. سطر <code dir="ltr">10</code> برچسب ندارد و سطر <code dir="ltr">13</code> ویژگی ندارد. اگر این جدول را بی‌دقت به یک مدل بدهید، دو نمونه یا خطا می‌دهند یا (بدتر) بعد از حذف <code dir="ltr">NaN</code>ها بی‌صدا کنار می‌روند. قبل از <code dir="ltr">concat</code> با <code dir="ltr">axis=1</code>، یا مطمئن شوید اندیس‌ها یکی است، یا هر دو را با <code dir="ltr">reset&#95;index(drop=True)</code> یکسان کنید (به شرطی که **ترتیب** سطرها واقعاً متناظر باشد).

</div>

<div dir="rtl">

## ۲. merge و join

### 💡 مفهوم

<code dir="ltr">concat</code> فقط چیدمان می‌کند. اما خیلی وقت‌ها می‌خواهیم دو جدول را بر اساس یک **کلید** مشترک به هم وصل کنیم: هر سفارش یک <code dir="ltr">customer&#95;id</code> دارد و می‌خواهیم نام و شهر مشتری را کنار سفارش بیاوریم. این همان **join** در پایگاه‌های داده است و در Pandas با <code dir="ltr">merge</code> انجام می‌شود.

پرسش اصلی این است: با سطرهایی که کلیدشان در جدول دیگر **پیدا نمی‌شود** چه کنیم؟ پارامتر <code dir="ltr">how</code> جواب را تعیین می‌کند:

| <code dir="ltr">how</code> | کدام کلیدها می‌مانند | معادل مجموعه‌ای (جلسه‌ی ۹) |
|---|---|---|
| <code dir="ltr">"inner"</code> (پیش‌فرض) | فقط کلیدهای موجود در **هر دو** جدول | اشتراک |
| <code dir="ltr">"left"</code> | همه‌ی کلیدهای جدول **چپ** | جدول چپ |
| <code dir="ltr">"right"</code> | همه‌ی کلیدهای جدول **راست** | جدول راست |
| <code dir="ltr">"outer"</code> | کلیدهای **هر کدام** از دو جدول | اجتماع |

</div>

<p align="center"><img src="images/s25_join_types.png" width="820" alt="s25_join_types.png"></p>

<p align="center" dir="rtl"><i>چهار نوع <code dir="ltr">merge</code>. این نمودار فقط می‌گوید کدام کلیدها می‌مانند؛ اگر یک کلید چند بار تکرار شده باشد، تعداد سطرها بیشتر از تعداد کلیدها می‌شود (ادامه‌ی همین بخش).</i></p>

In [ ]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name": ["Ali", "Sara", "Reza", "Mina"],
    "city": ["Bushehr", "Shiraz", "Bushehr", "Tehran"],
})
orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 2, 1, 5, 3],
    "amount": [250, 120, 80, 300, 60],
})

print(orders.merge(customers, on="customer_id"))
print()
print(orders.merge(customers, on="customer_id", how="left"))

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">orders.merge(customers, ...)</code> همان <code dir="ltr">pd.merge(orders, customers, ...)</code> است؛ <code dir="ltr">orders</code> جدول **چپ** و <code dir="ltr">customers</code> جدول **راست** است.
- مشتری <code dir="ltr">1</code> دو سفارش دارد؛ پس اطلاعات او در نتیجه **دو بار** تکرار شده است. این رابطه‌ی **چند به یک** (many-to-one) است: چند سفارش، یک مشتری. یک «جدول مرجع» (lookup) را به این شکل به جدول اصلی می‌چسبانیم.
- سفارش <code dir="ltr">104</code> مشتری <code dir="ltr">5</code> دارد که در جدول مشتری‌ها نیست. در <code dir="ltr">inner</code> این سفارش **حذف** شده (۴ سطر)، و در <code dir="ltr">left</code> مانده و نام و شهرش <code dir="ltr">NaN</code> است (۵ سطر).
- مشتری <code dir="ltr">4</code> (<code dir="ltr">Mina</code>) هیچ سفارشی ندارد؛ در هیچ‌کدام از این دو نتیجه دیده نمی‌شود.

### ❓ پیش‌بینی کن

قبل از اجرا: <code dir="ltr">how="right"</code> و <code dir="ltr">how="outer"</code> چند سطر می‌دهند؟ در هر کدام کجا <code dir="ltr">NaN</code> هست؟ چرا ستون‌های <code dir="ltr">order&#95;id</code> و <code dir="ltr">amount</code> در این دو حالت اعشاری می‌شوند؟

</div>

In [ ]:
for how in ["inner", "left", "right", "outer"]:
    print(how, len(orders.merge(customers, on="customer_id", how=how)))

print(orders.merge(customers, on="customer_id", how="outer"))

<div dir="rtl">

🔍 <code dir="ltr">right</code> پنج سطر دارد: چهار سطر مشترک به‌علاوه‌ی <code dir="ltr">Mina</code> بدون سفارش. <code dir="ltr">outer</code> شش سطر دارد: هر دو سطرِ بی‌جفت (سفارش <code dir="ltr">104</code> و مشتری <code dir="ltr">4</code>) را نگه می‌دارد. در سطر <code dir="ltr">Mina</code>، ستون‌های سفارش <code dir="ltr">NaN</code> گرفته‌اند و چون <code dir="ltr">NaN</code> از نوع <code dir="ltr">float</code> است، کل ستون‌های <code dir="ltr">order&#95;id</code> و <code dir="ltr">amount</code> به <code dir="ltr">float</code> تبدیل شده‌اند (<code dir="ltr">101.0</code>).

### 💡 پیدا کردن سطرهای بی‌جفت: <code dir="ltr">indicator</code>

با <code dir="ltr">indicator=True</code> یک ستون به نام <code dir="ltr">&#95;merge</code> اضافه می‌شود که برای هر سطر می‌گوید از کجا آمده: <code dir="ltr">both</code>، <code dir="ltr">left&#95;only</code> یا <code dir="ltr">right&#95;only</code>. این بهترین ابزار برای **کنترل کیفیت داده** بعد از هر <code dir="ltr">merge</code> است.

</div>

In [ ]:
check = orders.merge(customers, on="customer_id", how="outer", indicator=True)
print(check["_merge"].value_counts())
print(check.loc[check["_merge"] != "both", ["order_id", "customer_id", "name", "_merge"]])

<div dir="rtl">

🔍 یک سفارش با مشتری ناشناخته (<code dir="ltr">left&#95;only</code>) و یک مشتری بدون سفارش (<code dir="ltr">right&#95;only</code>). در داده‌ی واقعی، سطرهای <code dir="ltr">left&#95;only</code> معمولاً یعنی **خطای داده** (کلید اشتباه تایپ شده یا جدول مرجع ناقص است) و قبل از هر تحلیلی باید بررسی شوند.

### 💡 نام ستون‌های متفاوت و ستون‌های هم‌نام

- اگر نام ستون کلید در دو جدول فرق دارد، از <code dir="ltr">left&#95;on</code> و <code dir="ltr">right&#95;on</code> استفاده کنید.
- اگر غیر از کلید، ستون **هم‌نام** دیگری هم در دو جدول باشد، Pandas به آن‌ها پسوند <code dir="ltr">&#95;x</code> و <code dir="ltr">&#95;y</code> می‌دهد؛ با <code dir="ltr">suffixes</code> می‌توانید پسوندهای گویاتری بگذارید.

</div>

In [ ]:
products = pd.DataFrame({"pid": ["P1", "P2"], "price": [10, 25]})
lines = pd.DataFrame({"order_id": [1, 1, 2], "product": ["P1", "P2", "P2"], "qty": [3, 1, 2]})

full = lines.merge(products, left_on="product", right_on="pid")
full["total"] = full["qty"] * full["price"]
print(full)
print(full.groupby("order_id")["total"].sum())

a = pd.DataFrame({"id": [1, 2], "score": [0.8, 0.6]})
b = pd.DataFrame({"id": [1, 2], "score": [0.7, 0.9]})
print(a.merge(b, on="id", suffixes=("_model_a", "_model_b")))

<div dir="rtl">

🔍 با <code dir="ltr">left&#95;on</code> / <code dir="ltr">right&#95;on</code> هر دو ستون کلید (<code dir="ltr">product</code> و <code dir="ltr">pid</code>) در نتیجه می‌مانند؛ اگر یکی اضافه است، با <code dir="ltr">drop(columns="pid")</code> حذفش کنید. بعد از <code dir="ltr">merge</code>، محاسبه‌ی مبلغ هر سطر و جمع آن با <code dir="ltr">groupby</code> (جلسه‌ی ۲۴) الگوی بسیار رایجی است: سفارش <code dir="ltr">1</code> مبلغ <code dir="ltr">30 + 25 = 55</code> و سفارش <code dir="ltr">2</code> مبلغ <code dir="ltr">50</code> دارد.

### ⚠️ اشتباه رایج: کلید تکراری در هر دو جدول

اگر یک کلید در جدول چپ <code dir="ltr">m</code> بار و در جدول راست <code dir="ltr">n</code> بار تکرار شود، <code dir="ltr">merge</code> برای آن کلید **<code dir="ltr">m × n</code> سطر** می‌سازد (همه‌ی جفت‌ها). این «انفجار سطرها» بدون هیچ پیامی رخ می‌دهد و مثلاً جمع فروش را چند برابر نشان می‌دهد:

</div>

In [ ]:
left = pd.DataFrame({"k": ["x", "x", "y"], "v1": [1, 2, 3]})
right = pd.DataFrame({"k": ["x", "x", "z"], "v2": [10, 20, 30]})
print(left.merge(right, on="k"))

<div dir="rtl">

🔍 کلید <code dir="ltr">"x"</code> دو بار در هر طرف آمده و <code dir="ltr">2 × 2 = 4</code> سطر ساخته است؛ کلیدهای <code dir="ltr">"y"</code> و <code dir="ltr">"z"</code> جفتی ندارند و در <code dir="ltr">inner</code> حذف شده‌اند.

### 💡 محافظ: <code dir="ltr">validate</code>

اگر می‌دانید رابطه‌ی دو جدول چه باید باشد، آن را با <code dir="ltr">validate</code> اعلام کنید تا اگر داده با انتظار شما نخواند، به‌جای نتیجه‌ی غلط **خطا** بگیرید (ایده‌ی «بلند شکست بخور» در جلسه‌ی ۱۵):

| <code dir="ltr">validate</code> | یعنی کلید در کدام جدول باید یکتا باشد |
|---|---|
| <code dir="ltr">"one&#95;to&#95;one"</code> | هر دو |
| <code dir="ltr">"one&#95;to&#95;many"</code> | فقط چپ |
| <code dir="ltr">"many&#95;to&#95;one"</code> | فقط راست (مثل جدول مرجع مشتری‌ها) |
| <code dir="ltr">"many&#95;to&#95;many"</code> | هیچ‌کدام (بدون بررسی) |

</div>

In [ ]:
print(len(orders.merge(customers, on="customer_id", validate="many_to_one")))

left.merge(right, on="k", validate="many_to_one")

<div dir="rtl">

🔍 اولی درست است: در جدول مشتری‌ها هر <code dir="ltr">customer&#95;id</code> یک بار آمده است. دومی <code dir="ltr">MergeError</code> می‌دهد، چون کلید <code dir="ltr">"x"</code> در جدول راست تکراری است. می‌توانید این خطا را با <code dir="ltr">try</code> / <code dir="ltr">except</code> (جلسه‌ی ۱۵) بگیرید و گزارش کنید.

### 💡 <code dir="ltr">join</code>: ترکیب بر اساس اندیس

<code dir="ltr">DataFrame.join</code> شکل کوتاه <code dir="ltr">merge</code> است وقتی کلید جدول راست (و اختیاری جدول چپ) **اندیس** باشد. پیش‌فرض آن <code dir="ltr">how="left"</code> است (برخلاف <code dir="ltr">merge</code> که پیش‌فرضش <code dir="ltr">inner</code> است).

</div>

In [ ]:
cust = customers.set_index("customer_id")
print(orders.join(cust, on="customer_id"))

<div dir="rtl">

🔍 <code dir="ltr">on="customer&#95;id"</code> یعنی ستون <code dir="ltr">customer&#95;id</code> جدول چپ با **اندیس** <code dir="ltr">cust</code> مقایسه شود. نتیجه همان <code dir="ltr">merge</code> با <code dir="ltr">how="left"</code> است، با این تفاوت که اندیس <code dir="ltr">orders</code> حفظ می‌شود. برای وصل کردن یک جدول مرجع که اندیسش کلید است، <code dir="ltr">join</code> خواناتر است؛ در بقیه‌ی موارد <code dir="ltr">merge</code> صریح‌تر و انعطاف‌پذیرتر است.

</div>

<div dir="rtl">

## ۳. داده‌های زمانی

### 💡 مفهوم

بخش بزرگی از داده‌ها **زمان‌دار**ند: فروش روزانه، دمای ساعتی، تراکنش‌های بانکی، لاگ‌های سرور. در جلسه‌ی ۱۷ با <code dir="ltr">datetime</code> و <code dir="ltr">timedelta</code> آشنا شدید. Pandas همین مفهوم‌ها را برای **ستون‌ها و اندیس‌های کامل** فراهم می‌کند:

| مفهوم پایتون (جلسه‌ی ۱۷) | معادل Pandas |
|---|---|
| <code dir="ltr">datetime</code> | <code dir="ltr">Timestamp</code> (و ستون از نوع <code dir="ltr">datetime64</code>) |
| <code dir="ltr">timedelta</code> | <code dir="ltr">Timedelta</code> (و ستون از نوع <code dir="ltr">timedelta64</code>) |
| <code dir="ltr">strptime</code> | <code dir="ltr">pd.to&#95;datetime(..., format=...)</code> |

### 💡 تبدیل رشته به تاریخ

</div>

In [ ]:
raw = pd.Series(["2026-01-15", "2026-02-03", "15/03/2026", "not a date"])
dates = pd.to_datetime(raw, format="%Y-%m-%d", errors="coerce")
print(dates)

d = pd.to_datetime(pd.Series(["2026-01-15", "2026-02-03", "2026-03-21"]))
print(d.dt.year.tolist(), d.dt.month.tolist())
print(d.dt.dayofweek.tolist(), d.dt.day_name().tolist())
print(d.diff())
print((d.iloc[2] - d.iloc[0]).days)

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">format="%Y-%m-%d"</code> همان کدهای <code dir="ltr">strptime</code> (جلسه‌ی ۱۷) است. دادن قالب صریح هم سریع‌تر است و هم جلوی **حدس اشتباه** را می‌گیرد (آیا <code dir="ltr">03/04/2026</code> سوم آوریل است یا چهارم مارس؟).
- <code dir="ltr">errors="coerce"</code> هر مقداری را که با قالب نمی‌خواند، به جای خطا به <code dir="ltr">NaT</code> («Not a Time»، معادل زمانی <code dir="ltr">NaN</code>) تبدیل می‌کند؛ پس <code dir="ltr">"15/03/2026"</code> و <code dir="ltr">"not a date"</code> هر دو <code dir="ltr">NaT</code> شده‌اند. بدون آن، اولین مقدار نامعتبر کل تبدیل را با خطا متوقف می‌کرد. بعد از <code dir="ltr">coerce</code> همیشه بشمارید چند <code dir="ltr">NaT</code> دارید (<code dir="ltr">dates.isna().sum()</code>)؛ وگرنه داده را بی‌صدا از دست می‌دهید.
- <code dir="ltr">.dt</code> برای ستون‌های تاریخ همان کاری را می‌کند که <code dir="ltr">.str</code> برای ستون‌های رشته‌ای: دسترسی به اجزا (<code dir="ltr">year</code>، <code dir="ltr">month</code>، <code dir="ltr">day</code>، <code dir="ltr">dayofweek</code> که دوشنبه را <code dir="ltr">0</code> می‌شمارد، <code dir="ltr">day&#95;name()</code>).
- <code dir="ltr">diff</code> روی ستون تاریخ، <code dir="ltr">Timedelta</code> می‌دهد و <code dir="ltr">.days</code> تعداد روزهای آن است.

### 💡 سری زمانی: تاریخ به‌عنوان اندیس

وقتی اندیس یک <code dir="ltr">DataFrame</code> یا <code dir="ltr">Series</code> از نوع تاریخ باشد (**DatetimeIndex**)، کارهای زیادی ساده می‌شوند. <code dir="ltr">pd.date&#95;range</code> یک دنباله‌ی منظم از تاریخ‌ها می‌سازد:

</div>

In [ ]:
idx = pd.date_range("2026-01-01", periods=60, freq="D")
rng = np.random.default_rng(0)
sales = pd.Series(100 + rng.integers(0, 50, size=60), index=idx, name="sales")

print(sales.head())
print(sales["2026-02-01":"2026-02-05"])
print(len(sales.loc["2026-02"]))

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">freq="D"</code> یعنی روزانه. چند فرکانس رایج دیگر: <code dir="ltr">"h"</code> (ساعتی)، <code dir="ltr">"W"</code> (هفتگی، پایان هفته یکشنبه) و <code dir="ltr">"ME"</code> (پایان ماه). (در نسخه‌های قدیمی Pandas برای پایان ماه <code dir="ltr">"M"</code> نوشته می‌شد؛ در نسخه‌های جدید به‌جای آن <code dir="ltr">"ME"</code> بنویسید تا هشدار نگیرید.)
- با DatetimeIndex می‌توان با **رشته‌ی تاریخ** برش زد. برخلاف برش عددی، **هر دو سر** بازه شامل‌اند: <code dir="ltr">"2026-02-01":"2026-02-05"</code> پنج روز می‌دهد.
- <code dir="ltr">sales.loc["2026-02"]</code> همه‌ی روزهای فوریه را می‌دهد (۲۸ روز)؛ لازم نیست روز آخر ماه را بدانید.

### 💡 تغییر فرکانس: <code dir="ltr">resample</code>

<code dir="ltr">resample</code> شبیه <code dir="ltr">groupby</code> (جلسه‌ی ۲۴) است، اما گروه‌ها **بازه‌های زمانی** هستند. بعد از آن یک تابع تجمیع (<code dir="ltr">sum</code>، <code dir="ltr">mean</code>، ...) صدا می‌زنید:

</div>

In [ ]:
print(sales.resample("W").sum().head(3))
print(sales.resample("ME").mean().round(2))

<div dir="rtl">

🔍 اول ژانویه‌ی ۲۰۲۶ پنجشنبه است؛ پس اولین «هفته» (که به یکشنبه ۴ ژانویه ختم می‌شود) فقط **چهار روز** دارد و جمعش (<code dir="ltr">511</code>) با هفته‌های کامل بعدی قابل مقایسه نیست. در گزارش‌ها، بازه‌های ناقص ابتدا و انتها را حذف یا علامت‌گذاری کنید. برچسب هر گروه در <code dir="ltr">resample("ME")</code> **آخرین روز** ماه است.

### 💡 پنجره‌ی متحرک: <code dir="ltr">rolling</code>

<code dir="ltr">rolling(k)</code> برای هر نقطه، <code dir="ltr">k</code> مقدار **آخر** (شامل خود آن نقطه) را در نظر می‌گیرد؛ همان میانگین متحرکی که در جلسه‌ی ۱۷ با <code dir="ltr">deque</code> ساختید، این بار در یک خط:

</div>

In [ ]:
print(sales.rolling(7).mean().head(9).round(2))

<div dir="rtl">

🔍 شش مقدار اول <code dir="ltr">NaN</code> هستند، چون هنوز هفت مقدار جمع نشده است. میانگین متحرک نوسان روزانه را صاف می‌کند و **روند** را نشان می‌دهد (نمودار آن را در جلسه‌ی ۲۷ رسم می‌کنیم).

### 💡 مقایسه با گذشته: <code dir="ltr">shift</code>، <code dir="ltr">diff</code> و <code dir="ltr">pct&#95;change</code>

- <code dir="ltr">shift(1)</code> کل سری را یک گام به **جلو** می‌برد: مقدار هر روز، مقدار **دیروز** می‌شود. این ابزار اصلی ساختن **ویژگی‌های تأخیری (lag)** برای پیش‌بینی است.
- <code dir="ltr">diff()</code> برابر <code dir="ltr">x - x.shift(1)</code> است: تغییر نسبت به گام قبل.
- <code dir="ltr">pct&#95;change()</code> برابر <code dir="ltr">x / x.shift(1) - 1</code> است: تغییر **نسبی**.

</div>

In [ ]:
df = sales.to_frame()
df["lag1"] = df["sales"].shift(1)
df["diff"] = df["sales"].diff()
df["pct"] = df["sales"].pct_change().round(3)
print(df.head(4))

<div dir="rtl">

### 🔍 کد را بخوان

در روز دوم، <code dir="ltr">lag1</code> برابر فروش روز اول (<code dir="ltr">142</code>) است، <code dir="ltr">diff</code> برابر <code dir="ltr">131 - 142 = -11</code> و <code dir="ltr">pct</code> برابر <code dir="ltr">131 / 142 - 1 ≈ -0.077</code> (یعنی ۷٫۷٪ کاهش). روز اول برای هر سه <code dir="ltr">NaN</code> است، چون «دیروز»ی ندارد.

### ⚠️ اشتباه رایج: نشت اطلاعات آینده

وقتی می‌خواهید فروش **امروز** را پیش‌بینی کنید، هر ویژگی فقط باید از اطلاعاتی ساخته شود که **قبل از امروز** در دسترس بوده است. <code dir="ltr">sales.rolling(7).mean()</code> در روز <code dir="ltr">t</code> خود فروش روز <code dir="ltr">t</code> را هم در میانگین دارد؛ اگر آن را ویژگی مدل کنید، مدل در آموزش عالی به نظر می‌رسد اما در عمل (که فروش امروز را هنوز نمی‌دانید) شکست می‌خورد. نسخه‌ی درست: **اول <code dir="ltr">shift</code> و بعد <code dir="ltr">rolling</code>**: <code dir="ltr">sales.shift(1).rolling(7).mean()</code>. (این همان **نشت داده** است که در جلسه‌ی ۹ دیدید، این بار در بعد زمان.)

### ❓ پیش‌بینی کن

مقدار <code dir="ltr">"ME"</code> در کد زیر را به <code dir="ltr">"W"</code> تغییر دهید: خروجی چند سطر دارد؟ اولین و آخرین برچسب چه تاریخی است؟ (۶۰ روز از پنجشنبه ۱ ژانویه.)

</div>

In [ ]:
print(sales.resample("ME").sum())

<div dir="rtl">

🔍 با <code dir="ltr">"ME"</code> سه سطر می‌گیرید: ژانویه، فوریه و مارس (که فقط یک روز دارد؛ پس جمع آن با دو ماه دیگر قابل مقایسه نیست). با <code dir="ltr">"W"</code> نُه سطر می‌گیرید، با برچسب‌های یکشنبه ۴ ژانویه تا یکشنبه ۱ مارس. اولین هفته ناقص (چهار روز) است، اما آخرین هفته (۲۳ فوریه تا ۱ مارس) کامل است، چون روز آخر سری خودش یکشنبه است.

</div>

<div dir="rtl">

## 📌 آنچه باید بلد باشید

- <code dir="ltr">pd.concat</code> با <code dir="ltr">axis=0</code> (روی هم) و <code dir="ltr">axis=1</code> (کنار هم)؛ نقش <code dir="ltr">ignore&#95;index</code>، <code dir="ltr">keys</code> و <code dir="ltr">join="inner"</code>؛ چرا ستون‌های ناهمسان و اندیس‌های ناهماهنگ <code dir="ltr">NaN</code> می‌سازند.
- <code dir="ltr">merge</code> با <code dir="ltr">on</code>، <code dir="ltr">left&#95;on</code> / <code dir="ltr">right&#95;on</code>، <code dir="ltr">suffixes</code>؛ تعداد سطرهای <code dir="ltr">inner</code>، <code dir="ltr">left</code>، <code dir="ltr">right</code> و <code dir="ltr">outer</code> را پیش‌بینی کنید؛ چرا <code dir="ltr">NaN</code> ستون عدد صحیح را <code dir="ltr">float</code> می‌کند.
- کلید تکراری در دو طرف <code dir="ltr">m × n</code> سطر می‌سازد؛ <code dir="ltr">validate</code> برای اعلام رابطه‌ی مورد انتظار و <code dir="ltr">indicator=True</code> برای پیدا کردن سطرهای بی‌جفت.
- <code dir="ltr">join</code> برای ترکیب بر اساس اندیس (پیش‌فرض <code dir="ltr">how="left"</code>).
- <code dir="ltr">pd.to&#95;datetime</code> با <code dir="ltr">format</code> و <code dir="ltr">errors="coerce"</code>؛ <code dir="ltr">NaT</code>؛ ویژگی‌های <code dir="ltr">.dt</code>؛ <code dir="ltr">Timedelta</code>.
- DatetimeIndex: <code dir="ltr">date&#95;range</code>، برش با رشته‌ی تاریخ (هر دو سر شامل)، <code dir="ltr">loc["2026-02"]</code>.
- <code dir="ltr">resample</code> (با فرکانس‌های <code dir="ltr">"D"</code>، <code dir="ltr">"W"</code>، <code dir="ltr">"ME"</code>) و بازه‌های ناقص ابتدا و انتها.
- <code dir="ltr">rolling</code>، <code dir="ltr">shift</code>، <code dir="ltr">diff</code> و <code dir="ltr">pct&#95;change</code>؛ ساختن ویژگی‌های تأخیری؛ «اول <code dir="ltr">shift</code> بعد <code dir="ltr">rolling</code>» برای جلوگیری از نشت آینده.

</div>

<div dir="rtl">

## 📝 سؤالات مرور / نمونه سؤال امتحان

> اول خودتان جواب دهید (روی کاغذ، بدون اجرای کد)، بعد پاسخ را باز کنید و در آخر کد را اجرا کنید.

### سؤال ۱ — پیش‌بینی تعداد سطرها

ستون کلید دو جدول این است (بقیه‌ی ستون‌ها مهم نیستند):

</div>

```python
A = pd.DataFrame({"key": [1, 1, 2, 3], "a": [10, 11, 20, 30]})
B = pd.DataFrame({"key": [1, 2, 2, 4], "b": [100, 200, 201, 400]})
```

<div dir="rtl">

الف) <code dir="ltr">A.merge(B, on="key", how=h)</code> برای هر یک از <code dir="ltr">h</code> در <code dir="ltr">"inner"</code>، <code dir="ltr">"left"</code>، <code dir="ltr">"right"</code> و <code dir="ltr">"outer"</code> چند سطر دارد؟ روش شمارش خود را توضیح دهید.

ب) کدام‌یک از مقدارهای <code dir="ltr">validate</code> زیر خطا می‌دهند: <code dir="ltr">"one&#95;to&#95;one"</code>، <code dir="ltr">"one&#95;to&#95;many"</code>، <code dir="ltr">"many&#95;to&#95;one"</code>، <code dir="ltr">"many&#95;to&#95;many"</code>؟

ج) در نتیجه‌ی <code dir="ltr">how="outer"</code>، ستون <code dir="ltr">a</code> چه نوعی دارد؟ چرا؟

<details>
<summary><b>پاسخ</b></summary>

الف) برای هر کلید، تعداد سطرها حاصل‌ضرب تکرار آن در دو جدول است:

| کلید | تکرار در <code dir="ltr">A</code> | تکرار در <code dir="ltr">B</code> | سطرها |
|---|---|---|---|
| <code dir="ltr">1</code> | ۲ | ۱ | ۲ |
| <code dir="ltr">2</code> | ۱ | ۲ | ۲ |
| <code dir="ltr">3</code> | ۱ | ۰ | (بی‌جفت از چپ) |
| <code dir="ltr">4</code> | ۰ | ۱ | (بی‌جفت از راست) |

پس <code dir="ltr">inner = 4</code>، <code dir="ltr">left = 4 + 1 = 5</code>، <code dir="ltr">right = 4 + 1 = 5</code> و <code dir="ltr">outer = 4 + 1 + 1 = 6</code>.

ب) <code dir="ltr">"one&#95;to&#95;one"</code> خطا می‌دهد (کلید <code dir="ltr">1</code> در <code dir="ltr">A</code> و کلید <code dir="ltr">2</code> در <code dir="ltr">B</code> تکراری‌اند)، <code dir="ltr">"one&#95;to&#95;many"</code> خطا می‌دهد (<code dir="ltr">A</code> یکتا نیست)، <code dir="ltr">"many&#95;to&#95;one"</code> خطا می‌دهد (<code dir="ltr">B</code> یکتا نیست). فقط <code dir="ltr">"many&#95;to&#95;many"</code> بدون خطا اجرا می‌شود.

ج) <code dir="ltr">float64</code>: سطر کلید <code dir="ltr">4</code> در ستون <code dir="ltr">a</code> مقدار <code dir="ltr">NaN</code> دارد و <code dir="ltr">NaN</code> از نوع اعشاری است؛ پس کل ستون به <code dir="ltr">float</code> تبدیل می‌شود (<code dir="ltr">10.0</code>، <code dir="ltr">11.0</code>، ...).

</details>

### سؤال ۲ — <code dir="ltr">concat</code> و ستون‌های ناهمسان

دو بخش یک مجموعه‌داده از دو منبع مختلف جمع‌آوری شده‌اند:

</div>

```python
part1 = pd.DataFrame({"temp": [21.0, 23.5], "humidity": [40, 55]})
part2 = pd.DataFrame({"humidity": [60, 35, 50], "wind": [3.2, 5.1, 2.0]})

r1 = pd.concat([part1, part2])
r2 = pd.concat([part1, part2], join="inner", ignore_index=True)
```

<div dir="rtl">

الف) شکل (<code dir="ltr">shape</code>) <code dir="ltr">r1</code> و تعداد <code dir="ltr">NaN</code>های آن چیست؟ اندیس <code dir="ltr">r1</code> چیست؟

ب) <code dir="ltr">r2</code> چه ستون‌هایی و چند سطر دارد؟ در یک پروژه‌ی یادگیری ماشین، انتخاب بین این دو چه پیامدی دارد؟

ج) دانشجویی برای ساختن جدول آموزش، بعد از حذف سطرهای پرت از <code dir="ltr">X</code> (یک <code dir="ltr">DataFrame</code>)، آن را با بردار برچسب <code dir="ltr">y</code> (یک <code dir="ltr">Series</code> با اندیس <code dir="ltr">0</code> تا <code dir="ltr">n - 1</code>) این‌طور ترکیب کرده است: <code dir="ltr">pd.concat([X&#95;clean, y], axis=1)</code>. چه مشکلی پیش می‌آید و درستش چیست؟

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">r1.shape == (5, 3)</code> با ستون‌های <code dir="ltr">temp</code>، <code dir="ltr">humidity</code> و <code dir="ltr">wind</code>. سه سطر <code dir="ltr">part2</code> ستون <code dir="ltr">temp</code> ندارند و دو سطر <code dir="ltr">part1</code> ستون <code dir="ltr">wind</code> ندارند: <code dir="ltr">3 + 2 = 5</code> مقدار <code dir="ltr">NaN</code>. اندیس <code dir="ltr">r1</code> برابر <code dir="ltr">[0, 1, 0, 1, 2]</code> است (تکراری)، چون <code dir="ltr">ignore&#95;index</code> داده نشده.

ب) فقط ستون <code dir="ltr">humidity</code> و پنج سطر. <code dir="ltr">join="outer"</code> همه‌ی ویژگی‌ها را نگه می‌دارد اما <code dir="ltr">NaN</code> می‌سازد که باید جای آن‌ها را پر کرد (جلسه‌ی ۲۶)؛ <code dir="ltr">join="inner"</code> داده‌ی کامل اما با ویژگی‌های کمتر می‌دهد. انتخاب به این بستگی دارد که ویژگی‌های ازدست‌رفته چقدر مهم‌اند و پر کردن <code dir="ltr">NaN</code>ها چقدر قابل اعتماد است.

ج) <code dir="ltr">X&#95;clean</code> بعد از حذف سطرها اندیس‌های **پراکنده** دارد (مثلاً <code dir="ltr">0, 2, 3, 7, ...</code>)، اما <code dir="ltr">y</code> همه‌ی اندیس‌ها را دارد. <code dir="ltr">concat</code> با <code dir="ltr">axis=1</code> بر اساس اندیس جفت می‌کند؛ پس سطرهای حذف‌شده با ویژگی <code dir="ltr">NaN</code> برمی‌گردند. اگر برچسب‌ها را قبلاً بی‌دقت با <code dir="ltr">reset&#95;index</code> کرده باشید، بدتر هم می‌شود: برچسب هر سطر به سطر دیگری می‌چسبد، بدون هیچ خطایی. راه درست: همان سطرها را از <code dir="ltr">y</code> هم انتخاب کنید (<code dir="ltr">y.loc[X&#95;clean.index]</code>) یا از ابتدا فیلتر را روی جدولی انجام دهید که ویژگی و برچسب را با هم دارد.

</details>

### سؤال ۳ — کنترل کیفیت با <code dir="ltr">indicator</code>

با جدول‌های <code dir="ltr">orders</code> و <code dir="ltr">customers</code> بخش ۲، کد زیر را در نظر بگیرید:

</div>

```python
check = orders.merge(customers, on="customer_id", how="outer", indicator=True)

orphan_orders = check.loc[check["_merge"] == "left_only", "order_id"]
inactive = check.loc[check["_merge"] == "right_only", "name"]
revenue_by_city = (orders.merge(customers, on="customer_id", how="left")
                   .groupby("city")["amount"].sum())
```

<div dir="rtl">

الف) <code dir="ltr">orphan&#95;orders</code> و <code dir="ltr">inactive</code> چه مقدارهایی دارند؟

ب) <code dir="ltr">revenue&#95;by&#95;city</code> چیست؟ جمع کل آن را با <code dir="ltr">orders["amount"].sum()</code> مقایسه کنید. چرا برابر نیستند؟ کدام مبلغ «گم» شده است؟

ج) چطور می‌توانید سفارش‌های بی‌مشتری را در گزارش شهرها با برچسب <code dir="ltr">"Unknown"</code> نگه دارید؟

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">orphan&#95;orders</code> فقط شامل <code dir="ltr">104.0</code> است (اعشاری، چون در <code dir="ltr">outer</code> ستون <code dir="ltr">order&#95;id</code> به <code dir="ltr">float</code> تبدیل شده) و <code dir="ltr">inactive</code> فقط <code dir="ltr">"Mina"</code>.

ب) <code dir="ltr">Bushehr: 390</code> (<code dir="ltr">250 + 80 + 60</code>) و <code dir="ltr">Shiraz: 120</code>؛ جمع <code dir="ltr">510</code>. اما جمع کل سفارش‌ها <code dir="ltr">810</code> است. <code dir="ltr">groupby</code> به‌طور پیش‌فرض گروه <code dir="ltr">NaN</code> را **حذف** می‌کند؛ پس مبلغ <code dir="ltr">300</code> سفارش <code dir="ltr">104</code> (مشتری ناشناخته، شهر <code dir="ltr">NaN</code>) بی‌صدا از گزارش افتاده است. این دقیقاً از نوع خطاهایی است که <code dir="ltr">indicator</code> کمک می‌کند ببینید.

ج) قبل از <code dir="ltr">groupby</code>، جای خالی شهر را پر کنید:

<div dir="ltr">

```python
merged = orders.merge(customers, on="customer_id", how="left")
merged["city"] = merged["city"].fillna("Unknown")
print(merged.groupby("city")["amount"].sum())
```

</div>

حالا <code dir="ltr">Unknown: 300</code> هم در گزارش هست و جمع به <code dir="ltr">810</code> می‌رسد. (راه دیگر: <code dir="ltr">groupby("city", dropna=False)</code>.)

</details>

### سؤال ۴ — ویژگی‌های تأخیری برای چند فروشگاه

فروش روزانه‌ی دو فروشگاه در یک جدول بلند (long) آمده است:

</div>

```python
panel = pd.DataFrame({
    "store": ["A", "A", "A", "B", "B", "B"],
    "date": pd.to_datetime(["2026-01-01", "2026-01-02", "2026-01-03"] * 2),
    "sales": [10, 12, 15, 50, 45, 55],
})
panel = panel.sort_values(["store", "date"])
panel["lag_wrong"] = panel["sales"].shift(1)
panel["lag_right"] = panel.groupby("store")["sales"].shift(1)
panel["growth"] = panel.groupby("store")["sales"].pct_change()
```

<div dir="rtl">

الف) ستون‌های <code dir="ltr">lag&#95;wrong</code>، <code dir="ltr">lag&#95;right</code> و <code dir="ltr">growth</code> را بنویسید.

ب) <code dir="ltr">lag&#95;wrong</code> در کدام سطر اشتباه است و چرا این اشتباه برای یک مدل پیش‌بینی خطرناک است؟

ج) چرا <code dir="ltr">sort&#95;values</code> قبل از <code dir="ltr">shift</code> لازم است؟ اگر سطرهای جدول به ترتیب تصادفی بودند و این خط حذف می‌شد، چه اتفاقی می‌افتاد؟

<details>
<summary><b>پاسخ</b></summary>

الف)

| store | sales | <code dir="ltr">lag&#95;wrong</code> | <code dir="ltr">lag&#95;right</code> | <code dir="ltr">growth</code> |
|---|---|---|---|---|
| A | 10 | <code dir="ltr">NaN</code> | <code dir="ltr">NaN</code> | <code dir="ltr">NaN</code> |
| A | 12 | 10 | 10 | 0.2 |
| A | 15 | 12 | 12 | 0.25 |
| B | 50 | **15** | <code dir="ltr">NaN</code> | <code dir="ltr">NaN</code> |
| B | 45 | 50 | 50 | <code dir="ltr">-0.1</code> |
| B | 55 | 45 | 45 | 0.2222 |

ب) در اولین روز فروشگاه <code dir="ltr">B</code>، <code dir="ltr">lag&#95;wrong</code> فروش **آخرین روز فروشگاه <code dir="ltr">A</code>** (<code dir="ltr">15</code>) را «فروش دیروز» حساب کرده است. مدل رابطه‌ای را یاد می‌گیرد که وجود ندارد (فروش یک فروشگاه با فروش فروشگاه دیگر) و چون این خطا فقط در مرز گروه‌هاست، به‌راحتی دیده نمی‌شود. <code dir="ltr">groupby(...).shift</code> جابه‌جایی را **درون هر گروه** انجام می‌دهد.

ج) <code dir="ltr">shift</code> فقط **ترتیب فعلی سطرها** را می‌بیند و هیچ اطلاعی از ستون <code dir="ltr">date</code> ندارد. اگر سطرها مرتب نباشند، «سطر قبلی» لزوماً «روز قبلی» نیست و ویژگی تأخیری بی‌معنی (و گاهی از **آینده**) می‌شود.

</details>

### سؤال ۵ — <code dir="ltr">rolling</code> و نشت آینده

می‌خواهیم فروش هر روز را با «میانگین سه روز گذشته» پیش‌بینی کنیم:

</div>

```python
s = pd.Series([10, 20, 30, 40, 50],
              index=pd.date_range("2026-01-01", periods=5, freq="D"))

f_leaky = s.rolling(3).mean()
f_safe = s.shift(1).rolling(3).mean()
err_leaky = (s - f_leaky).abs().mean()
err_safe = (s - f_safe).abs().mean()
```

<div dir="rtl">

الف) <code dir="ltr">f&#95;leaky</code> و <code dir="ltr">f&#95;safe</code> را بنویسید.

ب) <code dir="ltr">err&#95;leaky</code> و <code dir="ltr">err&#95;safe</code> چقدرند؟ (<code dir="ltr">mean</code> مقدارهای <code dir="ltr">NaN</code> را نادیده می‌گیرد.) کدام عدد تصویر واقعی‌تری از عملکرد این روش پیش‌بینی در عمل می‌دهد؟

ج) یک ویژگی سوم پیشنهاد شده است: <code dir="ltr">s.rolling(3, center=True).mean()</code>. بدون اجرا بگویید آیا نشت دارد یا نه، و چرا.

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">f&#95;leaky = [NaN, NaN, 20, 30, 40]</code> و <code dir="ltr">f&#95;safe = [NaN, NaN, NaN, 20, 30]</code>.

ب) <code dir="ltr">err&#95;leaky = (&#124;30 - 20&#124; + &#124;40 - 30&#124; + &#124;50 - 40&#124;) / 3 = 10</code> و <code dir="ltr">err&#95;safe = (&#124;40 - 20&#124; + &#124;50 - 30&#124;) / 2 = 20</code>. <code dir="ltr">err&#95;safe</code> واقعی است: در زمان پیش‌بینی، فروش امروز را نمی‌دانید. <code dir="ltr">f&#95;leaky</code> یک‌سوم پاسخ درست را در خودش دارد و خطا را **کمتر از واقع** نشان می‌دهد. با یک مدل قوی‌تر (که می‌تواند سهم فروش امروز را از میانگین «باز کند») این خوش‌بینی بسیار بیشتر هم می‌شود.

ج) نشت دارد، و حتی بدتر: <code dir="ltr">center=True</code> پنجره را حول نقطه قرار می‌دهد؛ پس در روز <code dir="ltr">t</code> مقدارهای <code dir="ltr">t - 1</code>، <code dir="ltr">t</code> و <code dir="ltr">t + 1</code> را می‌گیرد، یعنی فروش **فردا** را هم در ویژگی دارد.

</details>

### سؤال ۶ — تقسیم زمانی آموزش و آزمون

داده‌ی روزانه‌ی یک فروشگاه از ۱ ژانویه تا ۳۰ آوریل ۲۰۲۶ داریم و می‌خواهیم مدلی بسازیم که فروش روزهای آینده را پیش‌بینی کند.

</div>

```python
days = pd.date_range("2026-01-01", "2026-04-30", freq="D")
data = pd.DataFrame({"sales": np.arange(len(days)) * 2 + 100}, index=days)

train = data.loc[:"2026-03-31"]
test = data.loc["2026-04-01":]
```

<div dir="rtl">

الف) <code dir="ltr">len(data)</code>، <code dir="ltr">len(train)</code> و <code dir="ltr">len(test)</code> چقدرند؟ آیا هیچ روزی در هر دو قرار دارد؟

ب) یک دانشجو به‌جای این کار، از تابع <code dir="ltr">train&#95;test&#95;split</code> تصادفی (جلسه‌ی ۸) با ۲۵٪ آزمون استفاده کرده است. دو دلیل بیاورید که چرا خطای آزمون او برای پیش‌بینی آینده **خوش‌بینانه** است.

ج) میانگین فروش در <code dir="ltr">train</code> و <code dir="ltr">test</code> چقدر است؟ این تفاوت درباره‌ی مدلی که فقط «میانگین داده‌ی آموزش» را پیش‌بینی کند چه می‌گوید؟

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">120</code>، <code dir="ltr">90</code> (ژانویه ۳۱ + فوریه ۲۸ + مارس ۳۱) و <code dir="ltr">30</code>. هیچ روز مشترکی نیست: برش با رشته‌ی تاریخ هر دو سر را شامل می‌شود، اما <code dir="ltr">"2026-03-31"</code> و <code dir="ltr">"2026-04-01"</code> دو روز متفاوت‌اند.

ب) (۱) روزهای آزمون **بین** روزهای آموزش پخش می‌شوند؛ مدل برای هر روز آزمون، روزهای قبل و **بعد** از آن را در آموزش دیده است و عملاً «درون‌یابی» می‌کند، نه پیش‌بینی آینده. (۲) ویژگی‌هایی مثل <code dir="ltr">lag</code> و <code dir="ltr">rolling</code> روزهای مجاور را به هم وصل می‌کنند؛ پس اطلاعات روزهای آزمون از طریق روزهای آموزشِ مجاور به مدل نشت می‌کند. در تقسیم زمانی، مدل فقط گذشته را می‌بیند؛ دقیقاً شرایطی که در عمل با آن روبه‌رو می‌شود.

ج) فروش در روز <code dir="ltr">i</code> برابر <code dir="ltr">100 + 2i</code> است؛ میانگین <code dir="ltr">train</code> (روزهای <code dir="ltr">0</code> تا <code dir="ltr">89</code>) برابر <code dir="ltr">100 + 2 × 44.5 = 189</code> و میانگین <code dir="ltr">test</code> (روزهای <code dir="ltr">90</code> تا <code dir="ltr">119</code>) برابر <code dir="ltr">100 + 2 × 104.5 = 309</code> است. مدلی که میانگین گذشته را پیش‌بینی می‌کند، روند صعودی را نمی‌بیند و روی آینده به‌طور **سیستماتیک** کم‌تخمین می‌زند (به‌طور متوسط <code dir="ltr">120</code> واحد). این مشکل در تقسیم تصادفی اصلاً دیده نمی‌شد.

</details>

### سؤال ۷ — <code dir="ltr">resample</code> و رشد ماهانه

</div>

```python
daily = pd.Series(range(1, 91),
                  index=pd.date_range("2026-01-01", periods=90, freq="D"))

monthly_total = daily.resample("ME").sum()
monthly_mean = daily.resample("ME").mean()
growth = monthly_total.pct_change()
```

<div dir="rtl">

الف) سه مقدار <code dir="ltr">monthly&#95;total</code> و <code dir="ltr">monthly&#95;mean</code> را حساب کنید. (راهنمایی: مقدار هر روز برابر شماره‌ی آن روز از ابتدای سال است.)

ب) <code dir="ltr">growth</code> را حساب کنید (تا سه رقم اعشار). مدیری از روی رشد ماهانه‌ی کل فروش نتیجه می‌گیرد که «فوریه نسبت به ژانویه رشد بیشتری از مارس نسبت به فوریه داشته است». با توجه به تعداد روزهای هر ماه، آیا این مقایسه منصفانه است؟ رشد را بر اساس <code dir="ltr">monthly&#95;mean</code> هم حساب کنید.

ج) <code dir="ltr">daily.resample("W").sum()</code> چند سطر دارد و مقدار سطر اول آن چیست؟

<details>
<summary><b>پاسخ</b></summary>

الف) ژانویه روزهای <code dir="ltr">1</code> تا <code dir="ltr">31</code>: جمع <code dir="ltr">496</code>، میانگین <code dir="ltr">16</code>. فوریه روزهای <code dir="ltr">32</code> تا <code dir="ltr">59</code>: جمع <code dir="ltr">(32 + 59) × 28 / 2 = 1274</code>، میانگین <code dir="ltr">45.5</code>. مارس روزهای <code dir="ltr">60</code> تا <code dir="ltr">90</code>: جمع <code dir="ltr">(60 + 90) × 31 / 2 = 2325</code>، میانگین <code dir="ltr">75</code>.

ب) <code dir="ltr">growth = [NaN, 1.569, 0.825]</code>. مقایسه‌ی جمع‌ها منصفانه نیست، چون فوریه ۲۸ روز و ماه‌های دیگر ۳۱ روز دارند؛ بخشی از تفاوت فقط به **طول ماه** برمی‌گردد. رشد میانگین روزانه: فوریه <code dir="ltr">45.5 / 16 - 1 ≈ 1.844</code> و مارس <code dir="ltr">75 / 45.5 - 1 ≈ 0.648</code>. نتیجه‌ی کیفی (رشد فوریه بیشتر است) این بار هم برقرار است، اما اعداد کاملاً عوض شده‌اند. برای مقایسه‌ی دوره‌های با طول متفاوت، **نرخ** (میانگین روزانه) گزارش کنید، نه جمع.

ج) ۱۴ سطر (از یکشنبه ۴ ژانویه تا یکشنبه ۵ آوریل). سطر اول جمع چهار روز اول است: <code dir="ltr">1 + 2 + 3 + 4 = 10</code>.

</details>

</div>

<div dir="rtl">

## خلاصه‌ی جلسه

- <code dir="ltr">concat</code> جدول‌ها را روی هم (<code dir="ltr">axis=0</code>) یا کنار هم (<code dir="ltr">axis=1</code>) می‌چیند و بر اساس نام ستون یا اندیس هم‌تراز می‌کند؛ ناهماهنگی، <code dir="ltr">NaN</code> می‌سازد.
- <code dir="ltr">merge</code> جدول‌ها را بر اساس کلید وصل می‌کند؛ <code dir="ltr">how</code> سرنوشت سطرهای بی‌جفت را تعیین می‌کند و کلید تکراری <code dir="ltr">m × n</code> سطر می‌سازد. همیشه با <code dir="ltr">validate</code> و <code dir="ltr">indicator</code> نتیجه را کنترل کنید.
- <code dir="ltr">join</code> شکل کوتاه <code dir="ltr">merge</code> بر اساس اندیس است.
- <code dir="ltr">to&#95;datetime</code>، <code dir="ltr">.dt</code> و DatetimeIndex کار با تاریخ را ساده می‌کنند؛ <code dir="ltr">resample</code> گروه‌بندی زمانی و <code dir="ltr">rolling</code> پنجره‌ی متحرک است.
- <code dir="ltr">shift</code>، <code dir="ltr">diff</code> و <code dir="ltr">pct&#95;change</code> ویژگی‌های زمانی می‌سازند؛ ویژگی‌ها فقط باید از گذشته ساخته شوند و تقسیم آموزش و آزمون در داده‌ی زمانی باید بر اساس زمان باشد.

**جلسه‌ی بعد:** پاک‌سازی و آماده‌سازی داده؛ مقدارهای گمشده (همان <code dir="ltr">NaN</code>هایی که امروز بعد از <code dir="ltr">concat</code> و <code dir="ltr">merge</code> دیدید)، داده‌های پرت، کدگذاری متغیرهای رسته‌ای و استانداردسازی.

</div>